In [ ]:
from datasets import load_dataset

ds = load_dataset("axmeu/wiki_fr")
train_texts = ds["train"]["text"]
test_texts = ds["test"]["text"][:10]
print(len(test_texts))

10


In [74]:
import sys
sys.path.append("..")
from src.BPE.naive import BPE
from src.BPE.fast import FastBPE
import time

bpe_naive = BPE.load("../results/models/bpe_fast_v32000_n180000.json")
bpe_fast  = FastBPE.load("../results/models/bpe_fast_v32000_n180000.json")

In [15]:
s = time.time()
encoded_naive = [bpe_naive.encode(text) for text in test_texts]
print(f"naive encoding in {time.time() - s}")

s = time.time()
encoded_fast = [bpe_fast.encode(text)  for text in test_texts]
print(f"fast encoding in {time.time() - s}")

naive encoding in 2.796447515487671
fast encoding in 0.012079715728759766


In [38]:
from collections import Counter
import regex

test_word = "con"

word_counts = Counter()
for text in train_texts:
    for word in regex.findall(r"[\p{L}\p{N}]+", text):
        word_counts[word] += 1

print(word_counts["con"])
print(bpe_naive.encode(test_word))

20
['con</w>']


In [75]:
def trace_word(word: str, merge_rules: list) -> None:
    tokens = list(word) + ["</w>"]
    print(f"Initial: {tokens}")
    
    rules_index = {pair: i for i, pair in enumerate(merge_rules)}
    
    step = 0
    while len(tokens) > 1:
        best_idx = None
        best_pos = None
        for i in range(len(tokens) - 1):
            rank = rules_index.get((tokens[i], tokens[i + 1]))
            if rank is not None and (best_idx is None or rank < best_idx):
                best_idx = rank
                best_pos = i
        
        if best_pos is None:
            break
        
        merged = tokens[best_pos] + tokens[best_pos + 1]
        
        print(f"Step {step+1:<4} "
              f"(rule {best_idx:<4}): "
              f"{tokens[best_pos]:<10} + "
              f"{tokens[best_pos+1]:<10} => "
              f"{merged:<20}"
            )

        tokens = tokens[:best_pos] + [merged] + tokens[best_pos + 2:]
        step += 1
    
    print(f"Final: {tokens}")

In [ ]:
trace_word("seulement", bpe_fast.merge_rules)

Initial: ['s', 'e', 'u', 'l', 'e', 'm', 'e', 'n', 't', '</w>']
Step 1    (rule 2   ): t          + </w>       => t</w>               
Step 2    (rule 4   ): e          + n          => en                  
Step 3    (rule 24  ): e          + u          => eu                  
Step 4    (rule 26  ): e          + m          => em                  
Step 5    (rule 27  ): en         + t</w>      => ent</w>             
Step 6    (rule 62  ): em         + ent</w>    => ement</w>           
Step 7    (rule 164 ): l          + ement</w>  => lement</w>          
Step 8    (rule 2103): eu         + lement</w> => eulement</w>        
Step 9    (rule 2152): s          + eulement</w> => seulement</w>       
Final: ['seulement</w>']
